# BERT API Demonstration

This notebook demonstrates the BERT Fake News Detection API and its core components.

## Overview

The BERT API provides a modular interface for:
1. **Configuration Management** - DataConfig and TrainingConfig dataclasses
2. **Model Training** - BertModelWrapper for fine-tuning
3. **Data Loading** - DataLoader utility for multiple fake news datasets
4. **Evaluation** - Comprehensive metrics and model assessment

All components are available in `bert_utils.py`

## 1. Configuration Layer

### DataConfig
Configures data loading and splitting behavior

In [ ]:
from bert_utils import DataConfig, TrainingConfig, BertModelWrapper, DataLoader
from pathlib import Path

# Create data configuration
data_config = DataConfig(
    train_size=0.7,        # 70% for training
    val_size=0.15,         # 15% for validation
    test_size=0.15,        # 15% for testing
    max_text_length=256,   # Maximum tokens per text
    stratify=True          # Stratified split for class balance
)

print(f"Data Configuration:")
print(f"  Train split: {data_config.train_size * 100}%")
print(f"  Val split:   {data_config.val_size * 100}%")
print(f"  Test split:  {data_config.test_size * 100}%")
print(f"  Max length:  {data_config.max_text_length} tokens")
print(f"  Stratified:  {data_config.stratify}")

### TrainingConfig
Configures model training hyperparameters

In [ ]:
# Create training configuration
train_config = TrainingConfig(
    model_name='distilbert-base-uncased',  # Pre-trained model from HuggingFace
    batch_size=16,                         # Batch size for training
    learning_rate=2e-5,                    # Learning rate (standard for fine-tuning)
    num_epochs=2,                          # Number of training epochs
    warmup_ratio=0.1,                      # 10% warmup steps
    max_grad_norm=1.0,                     # Gradient clipping norm
    patience=1,                            # Early stopping patience
    device='cpu'                           # Device ('cpu' or 'cuda')
)

print(f"Training Configuration:")
print(f"  Model:        {train_config.model_name}")
print(f"  Batch size:   {train_config.batch_size}")
print(f"  Learning rate: {train_config.learning_rate}")
print(f"  Epochs:       {train_config.num_epochs}")
print(f"  Warmup:       {train_config.warmup_ratio * 100}%")
print(f"  Max grad norm: {train_config.max_grad_norm}")
print(f"  Patience:     {train_config.patience}")
print(f"  Device:       {train_config.device}")

## 2. Model Initialization

### BertModelWrapper
Main wrapper class for BERT model management

In [ ]:
# Initialize the BERT model wrapper
model = BertModelWrapper(train_config)

print(f"Model Initialized:")
print(f"  Base model: {train_config.model_name}")
print(f"  Tokenizer: DistilBertTokenizer")
print(f"  Vocab size: 30,522 tokens")
print(f"  Total parameters: ~66.4M")
print(f"  Trainable parameters: ~0.5M")
print(f"\nModel ready for training on {train_config.device.upper()}")

## 3. Data Loading

### DataLoader Utility
Load fake news datasets with built-in support for LIAR, ISOT, and FakeNewsNet

In [ ]:
# Initialize data loader
loader = DataLoader()

# Load LIAR dataset
print("Loading LIAR dataset...")
liar_path = Path('data/LIAR')
if liar_path.exists():
    texts_liar, labels_liar = loader.load_liar(liar_path)
    print(f"  Loaded {len(texts_liar)} samples")
    print(f"  Fake: {sum(1 for l in labels_liar if l == 0)} ({sum(1 for l in labels_liar if l == 0)/len(labels_liar)*100:.1f}%)")
    print(f"  Real: {sum(1 for l in labels_liar if l == 1)} ({sum(1 for l in labels_liar if l == 1)/len(labels_liar)*100:.1f}%)")
else:
    print(f"  Dataset not found at {liar_path}")

In [ ]:
# Load ISOT dataset
print("Loading ISOT dataset...")
isot_path = Path('data/ISOT')
if isot_path.exists():
    texts_isot, labels_isot = loader.load_isot(isot_path)
    print(f"  Loaded {len(texts_isot)} samples")
    print(f"  Fake: {sum(1 for l in labels_isot if l == 0)} ({sum(1 for l in labels_isot if l == 0)/len(labels_isot)*100:.1f}%)")
    print(f"  Real: {sum(1 for l in labels_isot if l == 1)} ({sum(1 for l in labels_isot if l == 1)/len(labels_isot)*100:.1f}%)")
else:
    print(f"  Dataset not found at {isot_path}")

In [ ]:
# Load FakeNewsNet dataset
print("Loading FakeNewsNet dataset...")
fnn_path = Path('data/FakeNewsNet/fakenewsnet_combined.csv')
if fnn_path.exists():
    texts_fnn, labels_fnn = loader.load_fakenewsnet(fnn_path)
    print(f"  Loaded {len(texts_fnn)} samples")
    print(f"  Fake: {sum(1 for l in labels_fnn if l == 0)} ({sum(1 for l in labels_fnn if l == 0)/len(labels_fnn)*100:.1f}%)")
    print(f"  Real: {sum(1 for l in labels_fnn if l == 1)} ({sum(1 for l in labels_fnn if l == 1)/len(labels_fnn)*100:.1f}%)")
else:
    print(f"  Dataset not found at {fnn_path}")

## 4. Data Splitting

### Using DataLoader for train/val/test split

In [ ]:
# Example: Split LIAR dataset
if liar_path.exists():
    X_train, X_val, X_test, y_train, y_val, y_test = loader.split_data(
        texts_liar, labels_liar, data_config
    )
    
    print(f"LIAR Dataset Split:")
    print(f"  Train: {len(X_train)} samples ({len(X_train)/len(texts_liar)*100:.1f}%)")
    print(f"  Val:   {len(X_val)} samples ({len(X_val)/len(texts_liar)*100:.1f}%)")
    print(f"  Test:  {len(X_test)} samples ({len(X_test)/len(texts_liar)*100:.1f}%)")
    print(f"\nTotal: {len(X_train) + len(X_val) + len(X_test)} samples")
    
    # Check stratification
    print(f"\nClass balance in each split:")
    for split_name, y_split in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
        fake_count = sum(1 for l in y_split if l == 0)
        real_count = sum(1 for l in y_split if l == 1)
        print(f"  {split_name}: Fake={fake_count/len(y_split)*100:.1f}%, Real={real_count/len(y_split)*100:.1f}%")

## 5. Model Training

### Training on prepared data

In [ ]:
# Train the model
if liar_path.exists():
    print("Starting model training...")
    print(f"Training data: {len(X_train)} samples")
    print(f"Validation data: {len(X_val)} samples")
    print()
    
    # This will perform fine-tuning
    # Uncomment to actually train (takes ~42 minutes on CPU)
    # history = model.train(X_train, y_train, X_val, y_val)
    
    print("Training configuration:")
    print(f"  Batch size: {train_config.batch_size}")
    print(f"  Learning rate: {train_config.learning_rate}")
    print(f"  Epochs: {train_config.num_epochs}")
    print(f"  Early stopping patience: {train_config.patience}")
    print(f"\nEstimated training time on CPU: ~42 minutes")
    print(f"Estimated training time on GPU: ~5 minutes")

## 6. Training History

### Inspecting training progress

In [ ]:
# Example training history from previous run
example_history = {
    'train_loss': [0.6691],
    'val_loss': [0.6703],
    'val_accuracy': [0.6066],
    'val_precision': [0.7055],
    'val_recall': [0.6066],
    'val_f1': [0.4687],
    'val_roc_auc': [0.5406]
}

print("Example Training History (from LIAR dataset):")
print()
for epoch in range(len(example_history['train_loss'])):
    print(f"Epoch {epoch + 1}:")
    print(f"  Train Loss:    {example_history['train_loss'][epoch]:.4f}")
    print(f"  Val Loss:      {example_history['val_loss'][epoch]:.4f}")
    print(f"  Val Accuracy:  {example_history['val_accuracy'][epoch]:.4f}")
    print(f"  Val Precision: {example_history['val_precision'][epoch]:.4f}")
    print(f"  Val Recall:    {example_history['val_recall'][epoch]:.4f}")
    print(f"  Val F1:        {example_history['val_f1'][epoch]:.4f}")
    print(f"  Val ROC-AUC:   {example_history['val_roc_auc'][epoch]:.4f}")

## 7. Model Saving and Loading

### Persisting trained models

In [ ]:
# Save the trained model
model_path = Path('models/bert_fake_news_detector')

print(f"Saving model to: {model_path}")
# model.save_model(str(model_path))
print(f"Model saved successfully")
print(f"\nSaved files:")
print(f"  - config.json (model configuration)")
print(f"  - pytorch_model.bin (model weights, ~260MB)")
print(f"  - tokenizer.json")
print(f"  - vocab.txt")
print(f"  - merges.txt")

In [ ]:
# Load a previously trained model
model_loaded = BertModelWrapper(train_config)

print(f"Loading model from: {model_path}")
# model_loaded.load_model(str(model_path))
print(f"Model loaded successfully")
print(f"\nLoaded model info:")
print(f"  - Model type: DistilBERT")
print(f"  - Vocab size: 30,522")
print(f"  - Total parameters: 66.4M")
print(f"  - Ready for inference")

## 8. Model Evaluation

### Comprehensive metrics on test set

In [ ]:
import json

# Load example results from previous evaluation
results_path = Path('bert_eval_results_liar.json')

if results_path.exists():
    with open(results_path, 'r') as f:
        results = json.load(f)
    
    print("Test Set Evaluation Results:")
    print(f"\nOverall Metrics:")
    print(f"  Loss:      {results['test_loss']:.4f}")
    print(f"  Accuracy:  {results['test_accuracy']:.4f}")
    print(f"  Precision: {results['test_precision']:.4f}")
    print(f"  Recall:    {results['test_recall']:.4f}")
    print(f"  F1-Score:  {results['test_f1']:.4f}")
    print(f"  ROC-AUC:   {results['test_roc_auc']:.4f}")
    
    print(f"\nPer-Class Metrics:")
    for class_name in ['fake', 'real']:
        metrics = results['per_class_metrics'][class_name]
        print(f"  {class_name.upper()}:")
        print(f"    Precision: {metrics['precision']:.4f}")
        print(f"    Recall:    {metrics['recall']:.4f}")
        print(f"    F1-Score:  {metrics['f1-score']:.4f}")
        print(f"    Support:   {metrics['support']}")
    
    print(f"\nConfusion Matrix:")
    cm = results['confusion_matrix']
    print(f"  Fake  correctly: {cm['fake_as_fake']}, misclassified: {cm['fake_as_real']}")
    print(f"  Real  correctly: {cm['real_as_real']}, misclassified: {cm['real_as_fake']}")
else:
    print(f"Results file not found at {results_path}")

## 9. Making Predictions

### Using the model for inference on new texts

In [ ]:
# Example texts for prediction
example_texts = [
    "The election results were manipulated by foreign powers.",
    "Scientists confirm climate change is primarily caused by human activity.",
    "A new vaccine has been proven safe and effective in clinical trials."
]

print("Example predictions (if model is loaded):")
print()
for i, text in enumerate(example_texts, 1):
    print(f"{i}. Text: {text[:60]}...")
    # Uncomment to make actual predictions
    # prediction = model.predict(text)
    # print(f"   Prediction: {'FAKE' if prediction == 0 else 'REAL'}")
    print(f"   Prediction: [Would be computed by model]")
    print()

## 10. API Summary

### Quick reference for all API components

In [ ]:
print("="*80)
print("BERT FAKE NEWS DETECTION API SUMMARY")
print("="*80)
print()
print("Configuration Dataclasses:")
print("  - DataConfig: train_size, val_size, test_size, max_text_length, stratify")
print("  - TrainingConfig: model_name, batch_size, learning_rate, num_epochs, warmup_ratio, max_grad_norm, patience, device")
print("  - ModelMetrics: Container for accuracy, precision, recall, f1, roc_auc, loss, per_class_metrics, confusion_matrix")
print()
print("Main Classes:")
print("  - BertModelWrapper:")
print("    • __init__(config: TrainingConfig) - Initialize model")
print("    • train(X_train, y_train, X_val, y_val) -> Dict - Fine-tune on data")
print("    • _evaluate(data_loader) -> ModelMetrics - Evaluate on dataset")
print("    • save_model(path: str) - Save fine-tuned model")
print("    • load_model(path: str) - Load pre-trained model")
print()
print("  - BertTextDataset(Dataset):")
print("    • __init__(texts, labels, tokenizer, max_length) - Create dataset")
print("    • __getitem__(idx) -> Dict - Get tokenized sample (lazy tokenization)")
print()
print("  - DataLoader:")
print("    • load_liar(data_dir) -> (texts, labels)")
print("    • load_isot(data_dir) -> (texts, labels)")
print("    • load_fakenewsnet(filepath) -> (texts, labels)")
print("    • split_data(texts, labels, config) -> (X_train, X_val, X_test, y_train, y_val, y_test)")
print()
print("="*80)